# 🖼️ AI Image Captioner — Multi-Model, Multi-Source

**Author:** Mitadru  
**Status:** Beginner Portfolio Project  
**Last Updated:** November 2026

---

## What This Project Does

This notebook builds an **AI-powered image captioning tool** that can:

1. **Upload an image** from your computer and generate a caption
2. **Provide a URL** to a webpage and auto-caption every image on that page
3. **Switch between two open-source AI models** to compare results

### Models Used

| Model | Parameters | Strengths |
|-------|-----------|----------|
| [Salesforce/blip-image-captioning-base](https://huggingface.co/Salesforce/blip-image-captioning-base) | ~0.25B | Fast, great general captions |
| [microsoft/Florence-2-base](https://huggingface.co/microsoft/Florence-2-base) | ~0.23B | Detailed captions, multiple modes |

Both models are **completely free** and run on CPU (GPU recommended for speed).

### Why I Built This

I'm learning LLM / AI engineering and wanted a hands-on project that:
- Uses real open-source models from Hugging Face
- Combines web scraping with AI inference
- Has a clean Gradio UI anyone can try
- Runs anywhere — Windows, Linux, or Google Colab

---

## Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Import Libraries](#2-import-libraries)
3. [Load AI Models](#3-load-ai-models)
4. [Core Functions](#4-core-functions)
   - 4a. Caption a Single Image
   - 4b. Scrape & Caption Images from a URL
5. [Launch the Gradio App](#5-launch-the-gradio-app)
6. [What I Learned](#6-what-i-learned)

---

## 1. Setup & Installation

Run the cell below to install all dependencies.  
If you're on **Google Colab**, these will install into the Colab runtime.  
If you're **running locally**, make sure you have Python 3.9+ and (ideally) a virtual environment.

In [ ]:
# Install required packages (run once)
# On Colab, this cell installs everything you need.
# Locally, you can also run: pip install -r requirements.txt

!pip install -q transformers torch torchvision pillow \
    gradio requests beautifulsoup4 einops timm flash_attn

---

## 2. Import Libraries

We use:
- **transformers** — to load and run the AI models from Hugging Face
- **Pillow (PIL)** — for image processing
- **requests + BeautifulSoup** — for web scraping images from URLs
- **gradio** — to build a simple web UI
- **torch** — PyTorch, the deep learning framework under the hood

In [ ]:
import torch
import requests
import platform
import gradio as gr
import numpy as np
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from transformers import (
    AutoProcessor,
    BlipForConditionalGeneration,   # for BLIP
    AutoModelForCausalLM,            # for Florence-2
)

print(f"Python running on: {platform.system()} {platform.release()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## 3. Load AI Models

We load **both models** so the user can switch between them in the Gradio UI.

**Note:** The first run will download the model weights (~1-2 GB total).  
After that, they are cached locally and load much faster.

### How it works under the hood

```
Image  →  Processor (resize, normalize)  →  Model (generate tokens)  →  Decoder (tokens → text)
```

Both models follow this same pipeline, but their internals are different:
- **BLIP** uses a Vision Transformer encoder + text decoder
- **Florence-2** uses a DaVit vision encoder + a sequence-to-sequence language model

In [ ]:
# --- Device Setup ---
# Use GPU if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {device}")
print(f"Using dtype: {torch_dtype}")
print()

# =============================================
# Model 1: BLIP (Salesforce) — ~0.25B params
# =============================================
print("Loading BLIP model...")
blip_processor = AutoProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch_dtype,
).to(device)
print("✅ BLIP loaded!")

# =============================================
# Model 2: Florence-2 (Microsoft) — ~0.23B params
# =============================================
print("Loading Florence-2 model...")
florence_processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-base",
    trust_remote_code=True,
)
florence_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-base",
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)
print("✅ Florence-2 loaded!")

print("\n🎉 All models ready!")

---

## 4. Core Functions

This is where the main logic lives.  
We build two functions that power the Gradio app:

| Function | Input | Output |
|----------|-------|--------|
| `caption_image()` | An uploaded image + model choice | A text caption |
| `caption_from_url()` | A webpage URL + model choice | Captions for all images found |

### 4a. Caption a Single Image

This function takes an image (as a NumPy array from Gradio), converts it,
and runs it through the selected model to produce a caption.

In [ ]:
def generate_caption(pil_image, model_name):
    """
    Generate a caption for a single PIL Image.

    Args:
        pil_image:   A PIL Image object (RGB).
        model_name:  Which model to use — "BLIP" or "Florence-2".

    Returns:
        A string containing the generated caption.
    """
    if model_name == "BLIP":
        # BLIP uses a text prompt to guide captioning
        inputs = blip_processor(
            images=pil_image,
            text="the image of",
            return_tensors="pt",
        ).to(device, torch_dtype)

        output_ids = blip_model.generate(**inputs, max_new_tokens=50)
        caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)

    elif model_name == "Florence-2":
        # Florence-2 uses special task prompts like <CAPTION> or <DETAILED_CAPTION>
        task_prompt = "<DETAILED_CAPTION>"
        inputs = florence_processor(
            text=task_prompt,
            images=pil_image,
            return_tensors="pt",
        ).to(device, torch_dtype)

        output_ids = florence_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            num_beams=3,
            do_sample=False,
        )

        raw_text = florence_processor.batch_decode(
            output_ids, skip_special_tokens=False
        )[0]
        parsed = florence_processor.post_process_generation(
            raw_text,
            task=task_prompt,
            image_size=(pil_image.width, pil_image.height),
        )
        caption = parsed.get(task_prompt, raw_text)

    else:
        caption = "Error: Unknown model selected."

    return caption


def caption_image(input_image, model_name):
    """
    Gradio callback for the "Upload Image" tab.
    Converts the NumPy array from Gradio to a PIL image, then generates a caption.
    """
    if input_image is None:
        return "⚠️ Please upload an image first."

    pil_image = Image.fromarray(input_image).convert("RGB")
    caption = generate_caption(pil_image, model_name)
    return caption


print("✅ caption_image() defined.")

### 4b. Scrape & Caption Images from a URL

This function:
1. Fetches the HTML from a given URL
2. Finds all `<img>` tags using BeautifulSoup
3. Downloads each image
4. Runs the selected model on each image
5. Returns all captions in a formatted string

It also handles edge cases like:
- Relative URLs (`//` or `/`)
- SVG images (skipped, since these are vector graphics)
- Very small images like icons (skipped)

In [ ]:
def caption_from_url(url, model_name):
    """
    Scrape all images from a webpage URL and caption them.

    Args:
        url:         The full URL of the webpage to scrape (e.g. "https://en.wikipedia.org/wiki/Python").
        model_name:  Which model to use — "BLIP" or "Florence-2".

    Returns:
        A formatted string with all image URLs and their captions.
    """
    if not url or not url.strip():
        return "⚠️ Please enter a URL."

    url = url.strip()

    # --- Step 1: Fetch the webpage HTML ---
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        return f"❌ Could not fetch URL: {e}"

    # --- Step 2: Parse HTML and find all <img> tags ---
    soup = BeautifulSoup(response.text, "html.parser")
    img_elements = soup.find_all("img")

    if not img_elements:
        return "No images found on this page."

    # Figure out the base domain for fixing relative URLs
    from urllib.parse import urlparse
    parsed_url = urlparse(url)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    # --- Step 3: Loop through images, download, and caption ---
    results = []
    captioned_count = 0

    for idx, img_element in enumerate(img_elements, start=1):
        # Try to get the image URL from common attributes
        img_url = img_element.get("src") or img_element.get("data-src")
        if not img_url and img_element.has_attr("srcset"):
            img_url = img_element["srcset"].split()[0]
        if not img_url:
            continue

        # Skip SVGs (vector graphics, not photos)
        if ".svg" in img_url.lower():
            continue

        # Fix relative URLs
        if img_url.startswith("//"):
            img_url = "https:" + img_url
        elif img_url.startswith("/"):
            img_url = base_url + img_url
        elif not img_url.startswith("http"):
            continue

        # Download and caption the image
        try:
            img_response = requests.get(img_url, timeout=10, headers=headers)
            pil_image = Image.open(BytesIO(img_response.content))

            # Skip tiny images (icons, spacers, etc.)
            width, height = pil_image.size
            if width * height < 400:
                continue

            pil_image = pil_image.convert("RGB")
            caption = generate_caption(pil_image, model_name)

            captioned_count += 1
            results.append(f"📷 Image {captioned_count}:\n   URL: {img_url}\n   Caption: {caption}\n")

        except (OSError, Exception):
            # Skip images that can't be opened (corrupt, unsupported format, etc.)
            continue

    if not results:
        return "No captionable images found (images may be too small, SVGs, or inaccessible)."

    header = f"✅ Captioned {captioned_count} images from: {url}\nModel: {model_name}\n{'=' * 60}\n\n"
    return header + "\n".join(results)


print("✅ caption_from_url() defined.")

---

## 5. Launch the Gradio App

Now we bring it all together with a **Gradio UI** that has two tabs:

| Tab | What it does |
|-----|-------------|
| **Upload Image** | Upload a photo from your device → get a caption |
| **Caption from URL** | Paste a webpage URL → auto-caption all images on the page |

Both tabs let you **pick which model to use** via a dropdown.

### Running the app

- **Google Colab:** A public link will be generated automatically (`share=True`).
- **Locally:** The app opens at `http://127.0.0.1:7860`.

In [ ]:
# ============================================
# Build the Gradio Interface
# ============================================

# Detect if we're running in Google Colab
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# Model selector dropdown (reused in both tabs)
model_choices = ["BLIP", "Florence-2"]

# --- Tab 1: Upload Image ---
with gr.Blocks(
    title="AI Image Captioner",
    theme=gr.themes.Soft(),
) as app:

    gr.Markdown(
        """
        # 🖼️ AI Image Captioner
        Generate image captions using open-source AI models.  
        **Upload an image** or **paste a URL** to a webpage — the AI will describe what it sees.

        Models: **BLIP** (Salesforce, 0.25B) · **Florence-2** (Microsoft, 0.23B)  
        """
    )

    with gr.Tabs():

        # ---- Tab 1: Upload Image ----
        with gr.TabItem("📤 Upload Image"):
            with gr.Row():
                with gr.Column():
                    upload_image = gr.Image(
                        type="numpy",
                        label="Upload an image",
                    )
                    upload_model = gr.Dropdown(
                        choices=model_choices,
                        value="BLIP",
                        label="Select Model",
                    )
                    upload_btn = gr.Button("Generate Caption", variant="primary")

                with gr.Column():
                    upload_output = gr.Textbox(
                        label="Generated Caption",
                        lines=4,
                        show_copy_button=True,
                    )

            upload_btn.click(
                fn=caption_image,
                inputs=[upload_image, upload_model],
                outputs=upload_output,
            )

        # ---- Tab 2: Caption from URL ----
        with gr.TabItem("🌐 Caption from URL"):
            with gr.Row():
                with gr.Column():
                    url_input = gr.Textbox(
                        label="Webpage URL",
                        placeholder="https://en.wikipedia.org/wiki/Python_(programming_language)",
                        lines=1,
                    )
                    url_model = gr.Dropdown(
                        choices=model_choices,
                        value="BLIP",
                        label="Select Model",
                    )
                    url_btn = gr.Button("Scrape & Caption", variant="primary")

                with gr.Column():
                    url_output = gr.Textbox(
                        label="Captions",
                        lines=15,
                        show_copy_button=True,
                    )

            url_btn.click(
                fn=caption_from_url,
                inputs=[url_input, url_model],
                outputs=url_output,
            )

    gr.Markdown(
        """
        ---
        *Built with ❤️ using [Hugging Face Transformers](https://huggingface.co/docs/transformers)
        and [Gradio](https://gradio.app) · Open-source models, zero API costs.*
        """
    )

# Launch the app
# share=True creates a public URL (required for Colab)
app.launch(share=IS_COLAB)

---

## 6. What I Learned

### Key Takeaways

1. **Hugging Face `transformers` makes model loading simple** — just `from_pretrained()` with a model ID and you're running inference in a few lines.

2. **Open-source models are powerful** — BLIP and Florence-2 are both under 0.3B parameters but produce surprisingly good captions, even on CPU.

3. **Web scraping + AI is a useful combo** — being able to automatically caption every image on a webpage has real-world applications (accessibility, SEO, content moderation).

4. **Gradio is great for prototyping** — building a web UI took about 30 lines of code. It handles file uploads, text inputs, and even generates shareable links.

5. **Device management matters** — using `.to(device)` and `torch_dtype` lets the same code run on both GPU (fast) and CPU (slower but works everywhere).

### Possible Improvements

- Add batch processing for multiple uploaded images
- Support video frame captioning
- Fine-tune a model on a custom dataset
- Add object detection using Florence-2's `<OD>` task prompt
- Deploy as a standalone web app using Hugging Face Spaces

### Tech Stack

```
Python 3.9+
├── transformers     — Model loading & inference
├── torch            — Deep learning backend
├── Pillow           — Image processing
├── requests         — HTTP requests
├── beautifulsoup4   — HTML parsing
├── gradio           — Web UI
├── einops           — Tensor operations (Florence-2 dependency)
└── timm             — Vision model components (Florence-2 dependency)
```